# 02. LightGBM 베이스라인 학습 (Colab)

SPEC 8장. 로직은 전부 `ml/src` 에 있고, 이 노트북은 그 함수를 불러서 실행만 한다 (복붙 금지, SPEC 8.5).

1. Drive 마운트 → 2. 코드 준비 → 3. CSV 읽기 → 4. walk-forward 평가(모델 vs 베이스라인 3종)
→ 5. 최종 학습·산출물 저장(Drive) → 6. zip 다운로드 → 7. 로컬 `ml/models/` 에 풀고 `.env` 의 `MODEL_VERSION` 설정

**준비물**: 로컬에서 `make export-data` 로 만든 `data/*.csv` 를 Drive `MyDrive/StockSim/data/` 에 올려 둔다.
산출물·체크포인트는 Drive 에 저장되므로 세션이 끊겨도 남는다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/StockSim"
DATA_DIR = f"{BASE}/data"      # prices_daily.csv, index_daily.csv, macro_daily.csv
OUT_DIR = f"{BASE}/models"     # 산출물 저장 위치 (Drive)
REPO_DIR = "/content/stock-sim"

## 2. 코드 준비
둘 중 하나: (A) GitHub 에서 clone (비공개 저장소면 토큰 필요) / (B) 프로젝트 zip 을 Drive 에 올려 두고 풀기

In [ ]:
import os, subprocess, sys

REPO_URL = ""  # (A) 예: "https://github.com/<you>/<repo>.git" — 비워 두면 (B) 사용
REPO_ZIP = f"{BASE}/stock-sim.zip"  # (B)

if not os.path.exists(REPO_DIR):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["unzip", "-q", REPO_ZIP, "-d", "/content"], check=True)

# 저장소 안에서 ml/ 폴더 찾기 (zip 구조가 달라도 동작)
ML_ROOT = next(os.path.join(r, "ml") for r, d, _ in os.walk("/content") if "ml" in d and os.path.exists(os.path.join(r, "ml", "src", "features.py")))
sys.path.insert(0, ML_ROOT)
print("ml 경로:", ML_ROOT)

In [ ]:
%pip install -q lightgbm shap scikit-learn
import lightgbm, pandas as pd
from src.data import load_csv
from src import train_lgbm as T
from src.features import FEATURES
print("lightgbm", lightgbm.__version__, "| 피처", len(FEATURES))

## 3. 데이터 읽기

In [ ]:
prices, index, macro = load_csv(DATA_DIR)
print(f"가격 {len(prices):,}행, 종목 {prices.code.nunique()}개, {prices.date.min().date()} ~ {prices.date.max().date()}")
print("KOSPI", None if index is None else len(index), "| 거시", None if macro is None else len(macro))
data = T.make_dataset(prices, index, macro)
data[FEATURES].isna().mean().sort_values(ascending=False).head(10)  # 결측 비율 상위

## 4. walk-forward 평가 (학습 3년 / 검증 3개월 / 테스트 3개월, 5거래일 간격)
랜덤 분할 금지. 같은 테스트 구간에서 모델과 베이스라인(항상 상승, 5일 모멘텀, KOSPI 추종)을 비교한다.
베이스라인보다 나쁜 지표도 그대로 본다 (FR-PF-6).

In [ ]:
metrics, test_pred = T.walk_forward_eval(data)
table = pd.DataFrame(metrics["strategies"]).T
print("폴드 수:", len(metrics["folds"]))
table.round(4)

In [ ]:
import matplotlib.pyplot as plt
cal = pd.DataFrame(metrics["calibration_up"]).dropna()
plt.plot([0, 1], [0, 1], "--", color="gray", label="완벽한 보정")
plt.plot(cal["p_mean"], cal["actual"], "o-", label="모델")
plt.xlabel("예측 P(up)"); plt.ylabel("실제 상승 비율"); plt.legend(); plt.title("보정 곡선 (offline)"); plt.show()

## 5. 최종 학습과 산출물 저장 (Drive)

In [ ]:
path = T.run(prices, index, macro, OUT_DIR)
print("저장:", path)
print(sorted(os.listdir(path)))

## 6. zip 다운로드

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{os.path.basename(path)}", "zip", root_dir=os.path.dirname(path), base_dir=os.path.basename(path))
files.download(zip_path)

## 7. 로컬 적용
1. zip 을 프로젝트의 `ml/models/` 에 풀기 → `ml/models/<model_version>/...`
2. `.env` 에 `MODEL_VERSION=<model_version>` 설정 후 API 재시작
3. 서버는 시작할 때 `features.json` 이 서빙 피처와 다르면 기동을 거부한다 (학습-서빙 불일치 방지)